# 01｜金融市场与链上数据导入、清洗、日期匹配与质量检查
**负责人**：邓杰良（数据来源确认、原始数据留存、清洗、日期匹配、缺失值检查、数据字典与来源记录）  
**研究题目**：*Is Bitcoin Still the Virtual Gold? A Re-examination of Its Hedging Capabilities Using Updated Financial Market and On-Chain Data*  
**对应指南章节**：第 6 节（数据来源）与第 7 节（数据整理、日期匹配与质量检查）

### 使用说明（Google Colab / 本地 Jupyter）
1. 将原始下载文件放入 `02_Raw_Data/` 文件夹（**只读保存，不在原始文件上直接修改**）：
   - `BTC_Market_Price_Raw.json`（或 `.csv`）
   - `BTC_Transaction_Count_Raw.json`（或 `.csv`）
   - `FTSE100_Raw.xlsx`（或 `.csv`）
   - `USD_EUR_Raw.csv`
   - `USD_GBP_Raw.csv`
2. 按顺序运行本 Notebook 所有单元格。
3. 清洗后的共同样本数据将导出至 `03_Cleaned_Data/merged_clean_v01.csv`，处理日志导出至 `03_Cleaned_Data/cleaning_log_v01.csv`。


In [ ]:
# === 1. 环境初始化与路径配置 ===
import os
import json
import pathlib
import numpy as np
import pandas as pd

# 若在 Google Colab 中运行且文件存放在 Google Drive，可取消下方注释挂载云盘：
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = pathlib.Path('/content/drive/MyDrive/Blockchain_Finance_Project')

# 默认使用当前工作目录
BASE_DIR = pathlib.Path('.')
RAW_DIR = BASE_DIR / '02_Raw_Data'
CLEAN_DIR = BASE_DIR / '03_Cleaned_Data'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# 暂定样本区间（指南第 1.4 节：2016-01-01 至 2026-10-31）
SAMPLE_START = '2016-01-01'
SAMPLE_END = '2026-10-31'

print(f'RAW_DIR:   {RAW_DIR.resolve()}')
print(f'CLEAN_DIR: {CLEAN_DIR.resolve()}')
print(f'样本目标区间: {SAMPLE_START} 至 {SAMPLE_END}')


## 2. 定义各数据源标准化读取函数
严格保留原始单位，统一日期格式为 `YYYY-MM-DD`，数值字段转为 `float64`：
- **Blockchain.com JSON/CSV**：兼容 `{"values": [{"x": ts, "y": val}]}` 或以指标名命名的键（如 `"market-price"`, `"n-transactions"`），自动识别秒级/毫秒级时间戳。
- **FRED CSV (`DEXUSEU`, `DEXUSUK`)**：自动识别 `observation_date` 或 `DATE` 列，并将 FRED 缺失标记 `"."` 转为 `NaN`（**不随意前值填充**）。
- **Eikon for Datastream (`FTSE100_Raw.xlsx` / `.csv`)**：兼容常见日期列与收盘指数列，去除表头说明行与非交易日空值。


In [ ]:
def find_raw_file(raw_dir: pathlib.Path, stem: str, exts=('.json', '.csv', '.xlsx', '.xls')) -> pathlib.Path:
    for ext in exts:
        p = raw_dir / f'{stem}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'未找到 {stem} 对应的原始文件（支持扩展名: {exts}），请检查 {raw_dir}')

def load_blockchain_series(file_path: pathlib.Path, value_col_name: str) -> pd.DataFrame:
    """读取 Blockchain.com 下载的 JSON 或 CSV 原始文件。"""
    suffix = file_path.suffix.lower()
    if suffix == '.json':
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        # 寻找包含 {x, y} 列表的键
        records = None
        if isinstance(data, dict):
            if 'values' in data and isinstance(data['values'], list):
                records = data['values']
            else:
                for k, v in data.items():
                    if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict) and 'x' in v[0] and 'y' in v[0]:
                        records = v
                        break
        elif isinstance(data, list):
            records = data
        if records is None:
            raise ValueError(f'无法识别的 Blockchain.com JSON 结构: {file_path}')
        df = pd.DataFrame(records)
        # 判断时间戳单位（秒 vs 毫秒）
        ts_sample = pd.to_numeric(df['x'], errors='coerce').dropna().iloc[0]
        unit = 'ms' if ts_sample > 1e12 else 's'
        df['Date'] = pd.to_datetime(df['x'], unit=unit, utc=True).dt.strftime('%Y-%m-%d')
        df[value_col_name] = pd.to_numeric(df['y'], errors='coerce')
        df = df[['Date', value_col_name]]
    else:
        df = pd.read_csv(file_path)
        date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
        val_col = [c for c in df.columns if c != date_col][-1]
        df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
        df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
        df = df[['Date', value_col_name]]
    return df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

def load_fred_csv(file_path: pathlib.Path, value_col_name: str) -> pd.DataFrame:
    """读取 FRED 下载的 CSV 文件（DEXUSEU / DEXUSUK），将 '.' 转为 NaN。"""
    df = pd.read_csv(file_path, na_values=['.', 'NA', 'NaN', ''])
    date_col = [c for c in df.columns if 'date' in c.lower()][0]
    val_col = [c for c in df.columns if c != date_col][0]
    df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
    df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
    return df[['Date', value_col_name]].dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

def load_ftse_file(file_path: pathlib.Path, value_col_name: str = 'FTSE100_Level') -> pd.DataFrame:
    """读取 Datastream / Eikon 导出的 FTSE 100 Excel 或 CSV 文件。"""
    if file_path.suffix.lower() in ('.xlsx', '.xls'):
        df_raw = pd.read_excel(file_path)
    else:
        df_raw = pd.read_csv(file_path)
    # 尝试自动定位日期列与指数水平列
    df = df_raw.copy()
    date_candidates = [c for c in df.columns if any(k in str(c).lower() for k in ['date', 'time', '日期'])]
    date_col = date_candidates[0] if date_candidates else df.columns[0]
    val_candidates = [c for c in df.columns if c != date_col and any(k in str(c).lower() for k in ['close', 'price', 'index', 'ftse', 'pi', 'level'])]
    val_col = val_candidates[0] if val_candidates else df.columns[1]
    df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
    df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
    return df[['Date', value_col_name]].dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)


## 3. 读取五类原始数据并执行单序列质量检查
按照指南第 7.1 节与第 7.5 节检查：
1. 实际起止日期与总行数
2. 重复日期（若有，记录并保留最后一条或人工核查）
3. 缺失值（`NaN`）数量与占比
4. 非正数（`<= 0`）检查（对数变换要求所有价格、指数、汇率严格为正）
5. 极端日波动预警（例如单日涨跌幅绝对值超过 50% 时标出供人工复核，**不擅自删除真实极端行情**）


In [ ]:
cleaning_logs = []

def inspect_and_clean_series(df: pd.DataFrame, col_name: str, source_name: str) -> pd.DataFrame:
    df = df.copy()
    raw_rows = len(df)
    dup_count = int(df['Date'].duplicated().sum())
    if dup_count > 0:
        df = df.drop_duplicates(subset=['Date'], keep='last')
    
    # 截取暂定样本区间
    df_sub = df[(df['Date'] >= SAMPLE_START) & (df['Date'] <= SAMPLE_END)].copy()
    na_count = int(df_sub[col_name].isna().sum())
    non_pos_count = int((df_sub[col_name] <= 0).sum())
    
    # 剔除缺失值与非正值（仅记录，不插值填充）
    df_clean = df_sub.dropna(subset=[col_name])
    df_clean = df_clean[df_clean[col_name] > 0].sort_values('Date').reset_index(drop=True)
    
    # 极端变化检查（仅预警）
    pct_chg = df_clean[col_name].pct_change().abs()
    extreme_count = int((pct_chg > 0.50).sum())
    
    log_entry = {
        'Variable': col_name,
        'Source': source_name,
        'Raw_Rows_All': raw_rows,
        'Actual_Start': df_clean['Date'].min() if len(df_clean) else None,
        'Actual_End': df_clean['Date'].max() if len(df_clean) else None,
        'Sample_Rows_Valid': len(df_clean),
        'Duplicate_Dates_Removed': dup_count,
        'Missing_NA_Dropped': na_count,
        'Non_Positive_Dropped': non_pos_count,
        'Extreme_Jump_gt50pct_Flagged': extreme_count,
    }
    cleaning_logs.append(log_entry)
    print(f"[{col_name}] 有效观测: {len(df_clean)} | 区间: {log_entry['Actual_Start']} ~ {log_entry['Actual_End']} | "
          f"去重: {dup_count} | 缺失剔除: {na_count} | 非正值剔除: {non_pos_count} | >50%波动预警: {extreme_count}")
    return df_clean

btc_price_raw = load_blockchain_series(find_raw_file(RAW_DIR, 'BTC_Market_Price_Raw'), 'BTC_Price_USD')
btc_tx_raw    = load_blockchain_series(find_raw_file(RAW_DIR, 'BTC_Transaction_Count_Raw'), 'BTC_Tx_Count')
ftse_raw      = load_ftse_file(find_raw_file(RAW_DIR, 'FTSE100_Raw', ('.xlsx', '.xls', '.csv')), 'FTSE100_Level')
usdeur_raw    = load_fred_csv(find_raw_file(RAW_DIR, 'USD_EUR_Raw', ('.csv',)), 'USD_per_EUR')
usdgbp_raw    = load_fred_csv(find_raw_file(RAW_DIR, 'USD_GBP_Raw', ('.csv',)), 'USD_per_GBP')

btc_price = inspect_and_clean_series(btc_price_raw, 'BTC_Price_USD', 'Blockchain.com Market Price')
btc_tx    = inspect_and_clean_series(btc_tx_raw, 'BTC_Tx_Count', 'Blockchain.com Confirmed Tx')
ftse      = inspect_and_clean_series(ftse_raw, 'FTSE100_Level', 'Eikon/Datastream FTSE 100')
usdeur    = inspect_and_clean_series(usdeur_raw, 'USD_per_EUR', 'FRED DEXUSEU')
usdgbp    = inspect_and_clean_series(usdgbp_raw, 'USD_per_GBP', 'FRED DEXUSUK')


## 4. 交易日历匹配与链上指标对齐（指南第 7.3 节）
- **金融变量共同有效日期索引**：对 `BTC_Price_USD`、`FTSE100_Level`、`USD_per_EUR`、`USD_per_GBP` 取日期交集（Inner Join），**严禁对周末 FTSE 或汇率做前值填充**。
- **链上交易数量（`BTC_Tx_Count`）区间对齐**：Bitcoin 为每日 7 天连续产生区块，因此在两个相邻共同金融交易日 $(t-1, t]$ 之间可能包含周末或节假日：
  - `BTC_Tx_Count`：交易日 $t$ 当天的确认交易数；
  - `BTC_Tx_Count_Mean`：区间 $(t-1, t]$ 内每日确认交易数的**平均值**；
  - `BTC_Tx_Count_Sum`：区间 $(t-1, t]$ 内每日确认交易数的**累计值**。
  *(全组须在方法论章节中明确选定其中一种口径并保持论文与代码一致。)*


In [ ]:
# 1) 构建金融变量共同有效日期索引（Inner Join，不前值填充）
merged_fin = (
    btc_price
    .merge(ftse, on='Date', how='inner')
    .merge(usdeur, on='Date', how='inner')
    .merge(usdgbp, on='Date', how='inner')
    .sort_values('Date')
    .reset_index(drop=True)
)

# 2) 对齐链上交易数量（同日值 + 相邻共同交易日区间均值/累计值）
btc_tx_sorted = btc_tx.sort_values('Date').reset_index(drop=True)
common_dates = merged_fin['Date'].tolist()

tx_same_day = []
tx_interval_mean = []
tx_interval_sum = []
interval_days = []

tx_date_map = dict(zip(btc_tx_sorted['Date'], btc_tx_sorted['BTC_Tx_Count']))

for i, cur_d in enumerate(common_dates):
    tx_same_day.append(tx_date_map.get(cur_d, np.nan))
    if i == 0:
        tx_interval_mean.append(tx_date_map.get(cur_d, np.nan))
        tx_interval_sum.append(tx_date_map.get(cur_d, np.nan))
        interval_days.append(1)
    else:
        prev_d = common_dates[i - 1]
        mask = (btc_tx_sorted['Date'] > prev_d) & (btc_tx_sorted['Date'] <= cur_d)
        sub_vals = btc_tx_sorted.loc[mask, 'BTC_Tx_Count']
        interval_days.append(len(sub_vals))
        tx_interval_mean.append(sub_vals.mean() if len(sub_vals) > 0 else np.nan)
        tx_interval_sum.append(sub_vals.sum() if len(sub_vals) > 0 else np.nan)

merged_fin['BTC_Tx_Count'] = tx_same_day
merged_fin['BTC_Tx_Count_Mean'] = tx_interval_mean
merged_fin['BTC_Tx_Count_Sum'] = tx_interval_sum
merged_fin['Calendar_Days_In_Interval'] = interval_days

print(f"共同有效交易日样本量: {len(merged_fin)} 行")
print(f"共同样本起止日期: {merged_fin['Date'].min()} 至 {merged_fin['Date'].max()}")


## 5. 对数收益率与链上变化率计算（指南第 7.4 节）
在共同有效日期索引上计算：
$$r(t) = 100 \times [\ln(P(t)) - \ln(P(t-1))]$$
注意：第 1 行收益率为 `NaN`（无前一期），导出时保留完整对齐结构或由下游实证脚本 `dropna()` 处理。


In [ ]:
# 计算对数收益率（百分比形式，乘以 100）
for price_col, ret_col in [
    ('BTC_Price_USD', 'r_BTC'),
    ('FTSE100_Level', 'r_FTSE'),
    ('USD_per_EUR',   'r_USDEUR'),
    ('USD_per_GBP',   'r_USDGBP'),
]:
    merged_fin[ret_col] = 100.0 * (np.log(merged_fin[price_col]) - np.log(merged_fin[price_col].shift(1)))

# 链上交易数量的对数变化（百分比形式，乘以 100）
merged_fin['dln_BTC_Tx_Count'] = 100.0 * (np.log(merged_fin['BTC_Tx_Count']) - np.log(merged_fin['BTC_Tx_Count'].shift(1)))
merged_fin['dln_BTC_Tx_Count_Mean'] = 100.0 * (np.log(merged_fin['BTC_Tx_Count_Mean']) - np.log(merged_fin['BTC_Tx_Count_Mean'].shift(1)))

print("前 5 行预览：")
display_cols = ['Date', 'BTC_Price_USD', 'FTSE100_Level', 'USD_per_EUR', 'USD_per_GBP', 'BTC_Tx_Count', 'r_BTC', 'r_FTSE']
print(merged_fin[display_cols].head().to_string(index=False))


## 6. 质量验收检查与文件导出（指南第 7.5 节）


In [ ]:
# === 第 7.5 节验收检查 ===
assert merged_fin['Date'].is_unique, "错误：共同日期中存在重复值！"
assert merged_fin['Date'].is_monotonic_increasing, "错误：日期未按升序排列！"

core_cols = ['BTC_Price_USD', 'FTSE100_Level', 'USD_per_EUR', 'USD_per_GBP', 'BTC_Tx_Count']
missing_core = merged_fin[core_cols].isna().sum().to_dict()
print("核心字段缺失值统计（应全为 0，若 BTC_Tx_Count 有缺失需核查）:", missing_core)

# 导出清洗后数据与清洗日志
clean_csv_path = CLEAN_DIR / 'merged_clean_v01.csv'
log_csv_path = CLEAN_DIR / 'cleaning_log_v01.csv'

merged_fin.to_csv(clean_csv_path, index=False, encoding='utf-8-sig')
pd.DataFrame(cleaning_logs).to_csv(log_csv_path, index=False, encoding='utf-8-sig')

print(f"已导出清洗数据: {clean_csv_path} (共 {len(merged_fin)} 行, {len(merged_fin.columns)} 列)")
print(f"已导出清洗日志: {log_csv_path}")
